In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


nltk.download('punkt')
nltk.download('stopwords')

ModuleNotFoundError: No module named 'pandas'

In [13]:
# Loading dataset
data = pd.read_csv("./data.csv")

print(data.head())

                    subject  \
0             Special offer   
1  Job interview invitation   
2          Package delivery   
3  Urgent message from bank   
4                Newsletter   

                                                text label  
0  Get 20% off on your next order. Use code SAVE2...  spam  
1  We would like to invite you for an interview f...   ham  
2  Your package from Amazon has been delivered. P...   ham  
3  Action required: Your bank account has been te...  spam  
4  Here's our latest newsletter with updates on o...   ham  


In [14]:
# Clean text
def clean_text(text):
    text = text.lower()

    # replacing anything that is not alphanumeric to space
    text = re.sub('[^a-zA-Z0-9]', ' ', text)

    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))

    # only considering words that are not stopwords
    clean_words = [w for w in words if not w in stop_words]
    
    return ' '.join(clean_words)

In [17]:
# Applying the filter too dataset
data['text'] = data['text'].apply(clean_text)

In [19]:
# Feature extraction
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(data['text']).toarray()

In [20]:
# Label encoding replacing spam with 1 indicating postive for spam
y = np.array(data['label'].replace({'ham': 0, 'spam': 1}))

In [21]:
# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [22]:
# Model training
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [23]:
# Model evaluation
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)

Accuracy: 0.9090909090909091


In [25]:
# Test the model on a new email
def predict_email(filepath):
    email_text = ""
    # Read email from file
    with open(filepath) as f:
        email_text = f.read()

    # Preprocess the text
    email_text = clean_text(email_text)

    # Convert the text to a feature vector using the same CountVectorizer used for training
    email_vector = vectorizer.transform([email_text]).toarray()

    # Make a prediction using the pre-trained model
    prediction = model.predict(email_vector)

    # Print the result
    if prediction[0] == 0:
        print('The email is classified as HAM.')
    else:
        print('The email is classified as SPAM.')



In [27]:
# Test the model on a sample email
predict_email('test_email.txt')

The email is classified as SPAM.
